# RunPod Jupyter (Cloud-Only) Competition Notebook

This notebook is a full cloud workflow (no local steps):
- Install dependencies on RunPod
- Pull train/test data from Hugging Face
- Generate fast baseline submission (CLIP zero-shot)
- Train best-fit main model (SigLIP2) + TTA submission
- Optional DINOv2 + ensemble
- Upload submissions/models back to Hugging Face for online access

## 1) Config (edit once)

In [ ]:
from getpass import getpass

HF_TOKEN = getpass('Hugging Face token: ').strip()

# Competition pack source (GitHub URL).
# Example: https://github.com/your-user/competition_pack.git
COMP_PACK_GIT_URL = 'https://github.com/your-user/competition_pack.git'

# Folder names in /workspace
WORKSPACE_ROOT = '/workspace'
PROJECT_DIR = f'{WORKSPACE_ROOT}/competition_pack'
DATA_ROOT = f'{WORKSPACE_ROOT}/data'
TRAIN_DIR = f'{DATA_ROOT}/train'
TEST_DIR = f'{DATA_ROOT}/test'

# Hugging Face dataset repos (from your docs)
HF_TRAIN_DATASET = 'SmellsLikeAISpirit/plant-disease-train'
HF_TEST_DATASET = 'SmellsLikeAISpirit/plant-disease-test'

# Optional: repo to store outputs online (create if needed)
# Use format: username/repo-name
HF_OUTPUT_REPO = 'your-username/plant-competition-outputs'
HF_OUTPUT_REPO_TYPE = 'dataset'

print('Config loaded. Edit COMP_PACK_GIT_URL and HF_OUTPUT_REPO if needed.')

## 2) Environment setup on RunPod

In [ ]:
%%bash
set -euo pipefail
python -V
pip install --upgrade pip
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128
pip install transformers datasets huggingface_hub accelerate scikit-learn pillow timm open_clip_torch
nvidia-smi || true

In [ ]:
from huggingface_hub import login
import torch

login(token=HF_TOKEN)
print('HF login OK')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(props.total_memory / 1e9, 1))
else:
    print('WARNING: CUDA not available')

## 3) Get competition code + data (cloud-only)

In [ ]:
%%bash
set -euo pipefail
cd /workspace
if [ ! -d competition_pack ]; then
  git clone "$COMP_PACK_GIT_URL" competition_pack
fi
ls -la /workspace/competition_pack

## 4) Submission 1 (fast safety): CLIP zero-shot

In [ ]:
%%bash
set -euo pipefail
cd /workspace/competition_pack/advanced
python clip_zero_shot.py \
  --test-dir /workspace/data/test \
  --output-csv /workspace/submission_clip.csv
python ../final_submission_check.py \
  --csv /workspace/submission_clip.csv \
  --test-dir /workspace/data/test

## 5) Best-fit main strategy: SigLIP2 full fine-tune (skip phase1)

In [ ]:
%%bash
set -euo pipefail
cd /workspace/competition_pack/advanced
python train_vit_large.py \
  --model google/siglip2-so400m-patch14-384 \
  --data-dir /workspace/data/train \
  --max-images 0 \
  --batch-size 64 \
  --eval-batch-size 128 \
  --epochs 5 \
  --lr 1e-5 \
  --weighted-loss \
  --no-phase1 \
  --output-dir /workspace/siglip2-model

## 6) Submission 2: SigLIP2 + TTA + OOD threshold

In [ ]:
%%bash
set -euo pipefail
cd /workspace/competition_pack/advanced
python predict_tta.py \
  --model-dir /workspace/siglip2-model/best \
  --test-dir /workspace/data/test \
  --output-csv /workspace/submission_siglip2.csv \
  --num-tta 6 \
  --ood-threshold 0.5
python ../final_submission_check.py \
  --csv /workspace/submission_siglip2.csv \
  --test-dir /workspace/data/test \
  --model-dir /workspace/siglip2-model/best

## 7) Optional Submission 3: DINOv2 + ensemble

In [ ]:
%%bash
set -euo pipefail
cd /workspace/competition_pack/advanced
python train_dinov2.py \
  --model facebook/dinov2-large \
  --data-dir /workspace/data/train \
  --max-images 0 \
  --batch-size 32 \
  --epochs 5 \
  --lr 5e-6 \
  --grad-accum 2 \
  --output-dir /workspace/dinov2-model

In [ ]:
%%bash
set -euo pipefail
cd /workspace/competition_pack/advanced
python ensemble_predict.py \
  --model-dirs /workspace/siglip2-model/best /workspace/dinov2-model/best \
  --model-types hf dinov2 \
  --weights 0.6 0.4 \
  --test-dir /workspace/data/test \
  --output-csv /workspace/submission_ensemble.csv \
  --ood-threshold 0.5
python ../final_submission_check.py \
  --csv /workspace/submission_ensemble.csv \
  --test-dir /workspace/data/test \
  --model-dir /workspace/siglip2-model/best

## 8) Pick final submission.csv

In [ ]:
%%bash
set -euo pipefail
# Pick one: /workspace/submission_clip.csv, /workspace/submission_siglip2.csv, /workspace/submission_ensemble.csv
BEST=/workspace/submission_siglip2.csv
cp "$BEST" /workspace/submission.csv
ls -lh /workspace/submission*.csv
echo "Final submission file: /workspace/submission.csv"

## 9) Upload outputs to Hugging Face (online access)

In [ ]:
from datetime import datetime
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=HF_OUTPUT_REPO, repo_type=HF_OUTPUT_REPO_TYPE, private=True, exist_ok=True)

run_id = datetime.utcnow().strftime('%Y%m%d_%H%M%S')

# Upload CSV submissions
for p in [
    '/workspace/submission_clip.csv',
    '/workspace/submission_siglip2.csv',
    '/workspace/submission_ensemble.csv',
    '/workspace/submission.csv',
]:
    try:
        api.upload_file(
            path_or_fileobj=p,
            path_in_repo=f'{run_id}/' + p.split('/')[-1],
            repo_id=HF_OUTPUT_REPO,
            repo_type=HF_OUTPUT_REPO_TYPE,
        )
        print('Uploaded:', p)
    except Exception as e:
        print('Skip:', p, '|', e)

# Upload best model directories if they exist
for folder in ['/workspace/siglip2-model/best', '/workspace/dinov2-model/best']:
    try:
        api.upload_folder(
            folder_path=folder,
            path_in_repo=f'{run_id}/models/' + folder.split('/')[-2] + '/best',
            repo_id=HF_OUTPUT_REPO,
            repo_type=HF_OUTPUT_REPO_TYPE,
        )
        print('Uploaded folder:', folder)
    except Exception as e:
        print('Skip folder:', folder, '|', e)

print('Done.')
print('Repo URL:', f'https://huggingface.co/datasets/{HF_OUTPUT_REPO}')
print('This run path:', f'https://huggingface.co/datasets/{HF_OUTPUT_REPO}/tree/main/{run_id}')

## 10) Useful Hugging Face links

Data sources used in this notebook:
- Train dataset: https://huggingface.co/datasets/SmellsLikeAISpirit/plant-disease-train
- Test dataset: https://huggingface.co/datasets/SmellsLikeAISpirit/plant-disease-test

Model links:
- SigLIP2: https://huggingface.co/google/siglip2-so400m-patch14-384
- DINOv2 Large: https://huggingface.co/facebook/dinov2-large
- CLIP ViT-L/14: https://huggingface.co/openai/clip-vit-large-patch14

Your uploaded outputs (replace with your repo):
- https://huggingface.co/datasets/your-username/plant-competition-outputs